In [114]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

In [115]:
table_attribs = [
    'Name',
    'MC_USD_Billion',
    'MC_GBP_Billion',
    'MC_EUR_Billion',
    'MC_INR_Billion'
]

table_name = 'Largest_banks'
csv_path = './Largest_banks_data.csv'

df = pd.DataFrame(columns=table_attribs)

In [116]:

url = 'https://web.archive.org/web/20230908091635 /https://en.wikipedia.org/wiki/List_of_largest_banks'

In [117]:
def extract(url, table_attribs):
    page = requests.get(url).text
    data = BeautifulSoup(page, 'html.parser')
    
    rows_list = []
    tables = data.find_all('tbody')
    rows = tables[0].find_all('tr')

    for row in rows:
        col = row.find_all('td')
        if len(col) != 0:
            bank_name = col[1].text.strip()
            market_cap = float(col[2].text.strip())
            rows_list.append({'Name': bank_name, 'MC_USD_Billion': market_cap})

    df = pd.DataFrame(rows_list)
    return df

In [118]:
extract(url, table_attribs)

,Name,MC_USD_Billion
0,JPMorgan Chase,432.92
1,Bank of America,231.52
2,Industrial and Commercial Bank of China,194.56
3,Agricultural Bank of China,160.68
4,HDFC Bank,157.91
5,Wells Fargo,155.87
6,HSBC Holdings PLC,148.90
7,Morgan Stanley,140.83
8,China Construction Bank,139.82
9,Bank of China,136.81


In [119]:
df = extract(url, table_attribs)

In [120]:
import urllib.request

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-PY0221EN-Coursera/labs/v2/exchange_rate.csv"
urllib.request.urlretrieve(url, "exchange_rate.csv")

('exchange_rate.csv', <http.client.HTTPMessage at 0x206f1a11ed0>)

In [121]:
exchange_rate_csv = './exchange_rate.csv'  
output_csv_path = './transformed_data.csv'   

In [122]:
# Task 3: Transform
def transform(df, csv_path):
    exchange_rate_df = pd.read_csv(csv_path)

    dict_rates = exchange_rate_df.set_index('Currency').to_dict()['Rate']

    df['MC_GBP_Billion'] = (df['MC_USD_Billion'] * dict_rates['GBP']).round(2)
    df['MC_EUR_Billion'] = (df['MC_USD_Billion'] * dict_rates['EUR']).round(2)
    df['MC_INR_Billion'] = (df['MC_USD_Billion'] * dict_rates['INR']).round(2)

    return df

In [123]:
transform(df, exchange_rate_csv)

,Name,MC_USD_Billion,MC_GBP_Billion,MC_EUR_Billion,MC_INR_Billion
0,JPMorgan Chase,432.92,346.34,402.62,35910.71
1,Bank of America,231.52,185.22,215.31,19204.58
2,Industrial and Commercial Bank of China,194.56,155.65,180.94,16138.75
3,Agricultural Bank of China,160.68,128.54,149.43,13328.41
4,HDFC Bank,157.91,126.33,146.86,13098.63
5,Wells Fargo,155.87,124.70,144.96,12929.42
6,HSBC Holdings PLC,148.90,119.12,138.48,12351.26
7,Morgan Stanley,140.83,112.66,130.97,11681.85
8,China Construction Bank,139.82,111.86,130.03,11598.07
9,Bank of China,136.81,109.45,127.23,11348.39


In [124]:
df=transform(df, exchange_rate_csv)

In [125]:
# Task 4: Load to CSV
def load_to_csv(df, csv_path):
    df.to_csv(csv_path, index=False)

In [126]:
load_to_csv(df, output_csv_path)

In [127]:
# Task 5: Load to DB
def load_to_db(df, sql_connection, table_name):
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)

In [128]:
load_to_db(df, sqlite3.connect('banks_data.db'), table_name)

In [129]:
# Task 6: Run Query
def run_query(query_statement, sql_connection): # ✅ تصحيح run_query
    print(f'query : {query_statement}\n')
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output)

In [130]:
run_query("SELECT * FROM Largest_banks", sqlite3.connect('banks_data.db'))

query : SELECT * FROM Largest_banks

                                      Name  MC_USD_Billion  MC_GBP_Billion  \
0                           JPMorgan Chase          432.92          346.34   
1                          Bank of America          231.52          185.22   
2  Industrial and Commercial Bank of China          194.56          155.65   
3               Agricultural Bank of China          160.68          128.54   
4                                HDFC Bank          157.91          126.33   
5                              Wells Fargo          155.87          124.70   
6                        HSBC Holdings PLC          148.90          119.12   
7                           Morgan Stanley          140.83          112.66   
8                  China Construction Bank          139.82          111.86   
9                            Bank of China          136.81          109.45   

   MC_EUR_Billion  MC_INR_Billion  
0          402.62        35910.71  
1          215.31        19204.5

In [131]:

# Task 7: Verify Log File
def verify_log_file(log_file_path):
    with open(log_file_path, 'r') as file:
        print(file.read())

verify_log_file('code_log.txt')

2026-Sep-30-23:52:38 : Run Queries Phase Started
2026-Sep-30-23:52:38 : Executing Query 1: SELECT * FROM Largest_banks
2026-Sep-30-23:54:35 : Run Queries Phase Started
2026-Sep-30-23:54:35 : Executing Query 1: SELECT * FROM Largest_banks
2026-Sep-30-23:54:58 : Run Queries Phase Started
2026-Sep-30-23:54:58 : Executing Query 1: SELECT * FROM Largest_banks
2026-Sep-30-23:55:19 : Run Queries Phase Started
2026-Sep-30-23:55:19 : Executing Query 1: SELECT * FROM Largest_banks
2026-Sep-30-23:55:26 : Run Queries Phase Started
2026-Sep-30-23:55:26 : Executing Query 1: SELECT * FROM Largest_banks
2026-Sep-30-23:55:32 : Run Queries Phase Started
2026-Sep-30-23:55:32 : Executing Query 1: SELECT * FROM Largest_banks
2026-Sep-30-23:55:36 : Run Queries Phase Started
2026-Sep-30-23:55:36 : Executing Query 1: SELECT * FROM Largest_banks
2026-Sep-30-23:57:16 : Run Queries Phase Started
2026-Sep-30-23:57:16 : Executing Query 2: SELECT AVG(MC_GBP_Billion) FROM Largest_banks
2026-Sep-30-23:58:02 : Run Que